In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = None
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts"
pad_gdb = os.path.normpath(os.path.join(project_folder, r"Databases/Protected Areas/PADUS4_0_State_IL_GDB/PADUS4_0_StateIL.gdb"))
census_gdb = os.path.normpath(os.path.join(project_folder, "Wetland Mapping/TIGER_Line_Census.gdb"))
wetland_gdb = os.path.normpath(os.path.join(project_folder, "Wetland Mapping/Illinois_Wetland_Shapefiles.gdb"))

In [13]:
# target coordinate system: NAD 1983 Contiguous USA Albers — WKID 102003
target_sr = arcpy.SpatialReference(102003)

In [2]:
years = [2023, 2024]
n_y = len(years)

In [3]:
gap_layer = os.path.normpath(os.path.join(pad_gdb, "Step8_PADUS4_CombIL_ProDissolve"))

In [4]:
wetland_layers = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step11_{y}NLCD_AreaThreshold")) for y in years]

In [5]:
county_layer = os.path.normpath(os.path.join(census_gdb, "tl_2025_il_county_unclipped_NAD1983_Albers"))

# Clip GAP layer with NWI polgons

In [6]:
gap_clip_layers = [os.path.normpath(os.path.join(pad_gdb, f"Step9_PADUS4_CombIL_ProDissolve_NWI_Clip_{y}NLCD")) for y in years]

In [7]:
for i in range(n_y):
    arcpy.analysis.PairwiseClip(
        in_features = gap_layer,
        clip_features = wetland_layers[i],
        out_feature_class = gap_clip_layers[i]
    );

# Union GAP and NWI layers

In [8]:
gap_nwi_union_layers = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step12_{y}NLCD_Area_GAP_Union")) for y in years]

In [9]:
for i in range(n_y):
    arcpy.analysis.Union(
        in_features = [wetland_layers[i], 
                       gap_clip_layers[i]],
        out_feature_class = gap_nwi_union_layers[i]
    );

# Intersect GAP/NWI union with county layer

In [10]:
gap_nwi_union_cnty_int = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step13_{y}NLCD_GAP_Union_CntyIntersect")) for y in years]

In [11]:
for i in range(n_y):
    arcpy.analysis.PairwiseIntersect(
        in_features = [gap_nwi_union_layers[i], 
                       county_layer],
        out_feature_class = gap_nwi_union_cnty_int[i]
    );

# Recalculate areas

In [14]:
for i in range(n_y):
    # calculate PLANAR area
    arcpy.management.CalculateGeometryAttributes(
        in_features=gap_nwi_union_cnty_int[i],
        geometry_property=[["Polygon_Area_Ha_Planar", "AREA"]],
        area_unit="HECTARES",
        coordinate_system=target_sr,  # NAD 1983 Albers
        coordinate_format="SAME_AS_INPUT"                  
    );
    
    # calculate GEODESIC area
    arcpy.management.CalculateGeometryAttributes(
        in_features=gap_nwi_union_cnty_int[i],
        geometry_property=[["Polygon_Area_Ha_Geodesic", "AREA_GEODESIC"]],
        area_unit="HECTARES",
        coordinate_system=target_sr,
        coordinate_format="SAME_AS_INPUT"
    );

In [15]:
for i in range(n_y):
    # planar area
    arcpy.management.CalculateField(
        in_table=gap_nwi_union_cnty_int[i],
        field="Polygon_Area_Ac_Planar",
        expression="!Polygon_Area_Ha_Planar! * 2.47105",
        expression_type="PYTHON3"
    );

    # geodesic area
    arcpy.management.CalculateField(
        in_table=gap_nwi_union_cnty_int[i],
        field="Polygon_Area_Ac_Geodesic",
        expression="!Polygon_Area_Ha_Geodesic! * 2.47105", 
        expression_type="PYTHON3"
    );